In [ ]:
import pandas as pd
import digitalhub as dh

from data_preparation import *
from modelling import *

# Project creation

In [ ]:
project = dh.get_or_create_project("fair_hiring_tabular")

# Data preparation stage

In [ ]:
URL = "https://raw.githubusercontent.com/AlbanaCelepija/enhanced_mlops/refs/heads/main/framework/library/use_cases/tabular/src/local_platform/artifacts/data/recruitmentdataset-2022.csv"
di = project.new_dataitem(name="recruitmentdataset.csv",
                          kind="table",
                          path=URL)
di

In [ ]:
training_di = project.get_dataitem('recruitmentdataset.csv')
training_df = training_di.as_df()
training_df

In [ ]:
data_preparation_fn = project.new_function(name="data_preprocessing",
                                   kind="python",
                                   python_version="PYTHON3_10",
                                   code_src="mini_data_preparation.py",
                                   handler="load_data",
                                   requirements=["scikit-learn"])
data_preparation_fn = data_preparation_fn.run("job",wait=True)

In [ ]:
data_preparation_fn.output("dataset").as_df().head()

# Modelling stage

In [ ]:
train_fn = project.new_function(
    name="train-classifier",
    kind="python",
    python_version="PYTHON3_10",
    code_src="mini_modelling.py",
    handler="train_model",
    requirements=["scikit-learn", "numpy<2"],
)
dataset = data_preparation_fn.output("dataset")
train_run = train_fn.run(action="job", inputs={"data": dataset.key}, wait=True)

In [ ]:
model = train_run.output("model")
model.key

In [ ]:
from digitalhub import get_model

In [ ]:
model_obj = get_model(model.key)

## Evaluate performance metrics for each demographic group

In [ ]:
train_fn = project.new_function(
    name="train-classifier",
    kind="python",
    python_version="PYTHON3_10",
    code_src="mini_modelling.py",
    handler="train_model",
    requirements=["scikit-learn", "numpy<2"],
)
dataset = data_preparation_fn.output("dataset")
#train_run = train_fn.run(action="job", inputs={"data": dataset.key}, wait=True)

# Operationalisation stage

In [ ]:
serve_func = project.new_function(
    name="serve-classifier",
    kind="sklearnserve",
    path=model.key,
)
serve_run = serve_func.run("serve", wait=True)
serve_run

# Inference 

In [ ]:
test_df = data_preparation_fn.output("dataset").as_df().head()
test_df = test_df.drop(columns=["decision", "Id", "gender", "nationality"])
test_df


In [ ]:
data[4]

In [ ]:
data = test_df.values.tolist()
json_payload = {"inputs": [{"name": "input-0", "shape": [5, 23], "datatype": "FP32", "data": data}]}
json_payload

In [ ]:
serve_run.refresh()

In [ ]:
result = serve_run.refresh().invoke(json=json_payload).json()
print("Prediction result:")
print(result)

In [ ]:
result

# Workflow orchestration

In [ ]:
workflow = project.new_workflow(
    name="baseline-pipeline",
    kind="hera",
    code_src="pipeline.py",
    handler="pipeline",
)

In [ ]:
wf_build = workflow.run("build", wait=True)

In [ ]:
wf_run = workflow.run("pipeline", wait=True)